In [ ]:
# ============================================================================
# DIAGNÓSTICO: FN vs TN — O sinal existe nos dados antes de T?
# ============================================================================
# Pergunta central:
#   Os patches 7×7 de pixels que converteram (FN — modelo errou)
#   são distinguíveis dos patches de pixels que não converteram (TN)?
#
# Se SIM → erro de representação: o sinal existe, o modelo não lê
# Se NÃO → erro epistêmico: o sinal não está disponível antes de T
#
# Análises:
#   1. Distribuição de classes LULC nos patches (FN vs TN)
#   2. Heterogeneidade espacial (entropia do patch)
#   3. Presença de classes indicadoras de pressão agrícola
#   4. Evolução temporal do patch (T-5 a T-1)
#   5. Variação ano a ano (delta de classes)
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter
import rasterio
from rasterio.windows import Window
from tqdm.notebook import tqdm
from scipy import stats

# ── Caminhos ────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
DATA_DIR   = r"D:\Projetos\Cerrado\LULC"
FREEZE_DIR = BASE_DIR / "geofmv3b_frozen"
PRED_CSV   = BASE_DIR / "multihead_spatial_frozen" / "predictions_test.csv"

PATTERN   = "brazil_coverage_{ano}_Cerrado.tif"
YEARS     = list(range(1985, 2025))
PATCH_N   = 7
PATCH_YEARS = 5
NODATA_IN   = 255
NODATA_OUT  = 0

# Classes MapBiomas relevantes
LULC_LABELS = {
    0:  'NoData', 3: 'Forest', 4: 'Savanna', 11: 'Wetland',
    12: 'Grassland', 15: 'Pasture', 21: 'Mosaic Ag',
    39: 'Soybean', 41: 'Other Annual', 25: 'Urban',
    33: 'River/Lake'
}
# Classes indicadoras de pressão agrícola
AGRICULTURAL_CLASSES = {21, 39, 41, 20, 40, 62}

# Carregar predições
df = pd.read_csv(PRED_CSV)
print(f"✅ Predições carregadas: {len(df)} amostras")
print(f"   {df['error_type'].value_counts().to_dict()}")

# Focar em FN e TN — ambos são label=1 e label=0 respectivamente
df_FN = df[df['error_type'] == 'FN'].reset_index(drop=True)
df_TN = df[df['error_type'] == 'TN'].reset_index(drop=True)
print(f"\nFN (converteu, modelo errou): {len(df_FN)}")
print(f"TN (não converteu, acertou):  {len(df_TN)}")

In [ ]:
# ============================================================================
# EXTRAÇÃO DOS PATCHES — FN e TN
# ============================================================================
# Para cada pixel, extrair os patches 7×7 dos 5 anos antes de T.
# Amostragem balanceada: mesmo N de FN e TN para comparação justa.

def ler_patch(year, row, col, n=PATCH_N):
    path = os.path.join(DATA_DIR, PATTERN.format(ano=year))
    half = n // 2
    with rasterio.open(path) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)


def extrair_patches(df_subset, label_str, n_max=200):
    """
    Extrai patches (5, 7, 7) para cada pixel do subset.
    Retorna: patches (N, 5, 7, 7), T_values, rows, cols
    """
    n = min(n_max, len(df_subset))
    df_sample = df_subset.sample(n=n, random_state=42).reset_index(drop=True)

    patches = []
    Ts, rows_out, cols_out = [], [], []

    for _, r in tqdm(df_sample.iterrows(), total=n,
                     desc=f"Extraindo {label_str}"):
        row = int(r['row'])
        col = int(r['col'])
        T   = int(r['T'])

        anos_patch = [y for y in YEARS if y < T][-PATCH_YEARS:]
        patch = np.zeros((PATCH_YEARS, PATCH_N, PATCH_N), dtype=np.float32)
        for i, y in enumerate(anos_patch):
            patch[PATCH_YEARS - len(anos_patch) + i] = ler_patch(y, row, col)

        patches.append(patch)
        Ts.append(T)
        rows_out.append(row)
        cols_out.append(col)

    return np.array(patches), np.array(Ts), np.array(rows_out), np.array(cols_out)


# Balancear: usar min(len(FN), len(TN), 200) amostras de cada
N_SAMPLE = min(len(df_FN), len(df_TN), 200)
print(f"Extraindo {N_SAMPLE} patches de cada grupo...")
print("⚠️  Lê rasters — ~5-15 min")
print()

patches_FN, T_FN, rows_FN, cols_FN = extrair_patches(df_FN, 'FN', N_SAMPLE)
patches_TN, T_TN, rows_TN, cols_TN = extrair_patches(df_TN, 'TN', N_SAMPLE)

print(f"\n✅ Patches extraídos:")
print(f"   FN: {patches_FN.shape}")
print(f"   TN: {patches_TN.shape}")

In [ ]:
# ============================================================================
# ANÁLISE 1 — DISTRIBUIÇÃO DE CLASSES LULC NOS PATCHES
# ============================================================================
# Se FN e TN têm a mesma distribuição de classes → sinal ausente (epistêmico)
# Se FN tem mais classes agrícolas/transição → sinal existe (representação)

def class_distribution(patches):
    """Frequência de cada classe LULC em todos os patches."""
    flat = patches.flatten().astype(int)
    counter = Counter(flat)
    total = sum(counter.values())
    return {k: v/total for k, v in sorted(counter.items())}

dist_FN = class_distribution(patches_FN)
dist_TN = class_distribution(patches_TN)

# Classes presentes em qualquer dos dois
all_classes = sorted(set(dist_FN) | set(dist_TN))

print("Distribuição de classes LULC (% do total de pixels nos patches):")
print(f"{'Classe':<6} {'Nome':<18} {'FN':>8} {'TN':>8} {'Diferença':>10}")
print("-" * 55)
for cls in all_classes:
    fn_pct = dist_FN.get(cls, 0) * 100
    tn_pct = dist_TN.get(cls, 0) * 100
    diff   = fn_pct - tn_pct
    nome   = LULC_LABELS.get(cls, f'Class_{cls}')
    marker = ' ←' if abs(diff) > 1.0 else ''
    print(f"{cls:<6} {nome:<18} {fn_pct:>7.2f}% {tn_pct:>7.2f}% "
          f"{diff:>+9.2f}%{marker}")

# Presença de classes agrícolas
def pct_agricultural(patches):
    flat = patches.flatten().astype(int)
    return 100 * sum(1 for v in flat if v in AGRICULTURAL_CLASSES) / len(flat)

ag_FN = pct_agricultural(patches_FN)
ag_TN = pct_agricultural(patches_TN)
print(f"\nPresença de classes agrícolas (21, 39, 41...):")
print(f"   FN: {ag_FN:.2f}%")
print(f"   TN: {ag_TN:.2f}%")
print(f"   Diferença: {ag_FN-ag_TN:+.2f}pp")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras comparativas
classes_plot = [c for c in all_classes
                if dist_FN.get(c,0)*100 > 0.1 or dist_TN.get(c,0)*100 > 0.1]
x = np.arange(len(classes_plot))
w = 0.35
axes[0].bar(x - w/2, [dist_FN.get(c,0)*100 for c in classes_plot],
            w, label='FN (converteu, errou)', color='#c0392b', alpha=0.8)
axes[0].bar(x + w/2, [dist_TN.get(c,0)*100 for c in classes_plot],
            w, label='TN (não converteu, acertou)', color='#2980b9', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    [LULC_LABELS.get(c, str(c)) for c in classes_plot],
    rotation=45, ha='right')
axes[0].set_ylabel('% pixels nos patches')
axes[0].set_title('Distribuição de classes LULC: FN vs TN')
axes[0].legend()

# Diferença
diffs = [dist_FN.get(c,0)*100 - dist_TN.get(c,0)*100 for c in classes_plot]
colors = ['#c0392b' if d > 0 else '#2980b9' for d in diffs]
axes[1].bar(x, diffs, color=colors, alpha=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    [LULC_LABELS.get(c, str(c)) for c in classes_plot],
    rotation=45, ha='right')
axes[1].set_ylabel('Diferença FN − TN (pp)')
axes[1].set_title('Diferença de classes: vermelho=mais em FN')

plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'fn_tn_class_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: fn_tn_class_distribution.png")

In [ ]:
# ============================================================================
# ANÁLISE 2 — HETEROGENEIDADE ESPACIAL DOS PATCHES
# ============================================================================
# Entropia de Shannon por patch: H = -sum(p_i * log(p_i))
# Alta entropia → patch heterogêneo (muitas classes diferentes)
# Baixa entropia → patch homogêneo (pastagem uniforme)
# Se FN tem maior entropia → há sinal espacial que o modelo não captura

def patch_entropy(patch_5x7x7):
    """Entropia de Shannon média dos 5 frames de um patch."""
    entropies = []
    for t in range(patch_5x7x7.shape[0]):
        frame = patch_5x7x7[t].flatten().astype(int)
        counts = np.bincount(frame, minlength=50)
        probs  = counts[counts > 0] / len(frame)
        H = -np.sum(probs * np.log(probs + 1e-10))
        entropies.append(H)
    return np.mean(entropies)


def patch_n_classes(patch_5x7x7):
    """Número de classes únicas no patch (excluindo 0=NoData)."""
    return len(set(patch_5x7x7.flatten().astype(int)) - {0})


def patch_center_diff(patch_5x7x7):
    """Fração de vizinhos com classe diferente do pixel central (T-1)."""
    frame = patch_5x7x7[-1]  # último ano (T-1)
    center_class = frame[PATCH_N//2, PATCH_N//2]
    flat = frame.flatten().astype(int)
    # Excluir centro e zeros
    neighbors = [v for i, v in enumerate(flat) if i != PATCH_N*PATCH_N//2 and v > 0]
    if not neighbors: return 0.0
    return sum(1 for v in neighbors if v != center_class) / len(neighbors)


# Calcular métricas para todos os patches
print("Calculando métricas de heterogeneidade...")

ent_FN = np.array([patch_entropy(p) for p in patches_FN])
ent_TN = np.array([patch_entropy(p) for p in patches_TN])

ncls_FN = np.array([patch_n_classes(p) for p in patches_FN])
ncls_TN = np.array([patch_n_classes(p) for p in patches_TN])

cdiff_FN = np.array([patch_center_diff(p) for p in patches_FN])
cdiff_TN = np.array([patch_center_diff(p) for p in patches_TN])

# Teste estatístico (Mann-Whitney — não assume normalidade)
def mw_test(a, b, metric_name):
    stat, pval = stats.mannwhitneyu(a, b, alternative='two-sided')
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {metric_name:<30} FN={a.mean():.4f} TN={b.mean():.4f} "
          f"p={pval:.4f} {sig}")
    return pval

print(f"\nTeste Mann-Whitney (FN vs TN):")
print(f"  {'Métrica':<30} {'FN media':>10} {'TN media':>10} {'p-valor':>10} {'Sig':>5}")
print("-" * 65)
p_ent  = mw_test(ent_FN,   ent_TN,   'Entropia Shannon')
p_ncls = mw_test(ncls_FN,  ncls_TN,  'N classes únicas')
p_cd   = mw_test(cdiff_FN, cdiff_TN, 'Frac vizinhos dif. do centro')

# Gráfico
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, fn_vals, tn_vals, title, xlabel in [
    (axes[0], ent_FN,   ent_TN,   'Entropia do patch', 'Shannon entropy'),
    (axes[1], ncls_FN,  ncls_TN,  'N classes únicas',  'N classes'),
    (axes[2], cdiff_FN, cdiff_TN, 'Fração vizinhos\ndiferentes do centro', 'Fração'),
]:
    ax.hist(fn_vals, bins=20, alpha=0.6, color='#c0392b',
            label=f'FN (n={len(fn_vals)})', density=True)
    ax.hist(tn_vals, bins=20, alpha=0.6, color='#2980b9',
            label=f'TN (n={len(tn_vals)})', density=True)
    ax.axvline(fn_vals.mean(), color='#c0392b', linestyle='--', linewidth=2)
    ax.axvline(tn_vals.mean(), color='#2980b9', linestyle='--', linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Densidade')
    ax.legend()

plt.suptitle('Heterogeneidade espacial dos patches: FN vs TN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'fn_tn_heterogeneity.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: fn_tn_heterogeneity.png")

In [ ]:
# ============================================================================
# ANÁLISE 3 — EVOLUÇÃO TEMPORAL DO PATCH (T-5 a T-1)
# ============================================================================
# O patch muda ao longo dos 5 anos antes de T?
# Se FN mostra mais mudança temporal que TN → há sinal temporal no patch

def temporal_change(patch_5x7x7):
    """
    Mudança temporal: fração de pixels que mudaram de classe
    de um ano para o próximo, média sobre todos os pares de anos.
    """
    changes = []
    for t in range(1, patch_5x7x7.shape[0]):
        frame_curr = patch_5x7x7[t].flatten()
        frame_prev = patch_5x7x7[t-1].flatten()
        # Considerar apenas pixels com dados
        valid = (frame_curr > 0) & (frame_prev > 0)
        if valid.sum() == 0: continue
        changed = (frame_curr[valid] != frame_prev[valid]).mean()
        changes.append(changed)
    return np.mean(changes) if changes else 0.0


def agricultural_trend(patch_5x7x7):
    """
    Tendência de aumento de classes agrícolas ao longo dos 5 anos.
    Slope positivo = crescimento de pressão agrícola na vizinhança.
    """
    ag_per_year = []
    for t in range(patch_5x7x7.shape[0]):
        frame = patch_5x7x7[t].flatten().astype(int)
        ag_frac = sum(1 for v in frame
                      if v in AGRICULTURAL_CLASSES) / len(frame)
        ag_per_year.append(ag_frac)
    # Slope da regressão linear
    if len(ag_per_year) < 2: return 0.0
    x = np.arange(len(ag_per_year))
    slope, _ = np.polyfit(x, ag_per_year, 1)
    return slope


print("Calculando evolução temporal...")

change_FN = np.array([temporal_change(p)     for p in patches_FN])
change_TN = np.array([temporal_change(p)     for p in patches_TN])
trend_FN  = np.array([agricultural_trend(p)  for p in patches_FN])
trend_TN  = np.array([agricultural_trend(p)  for p in patches_TN])

print(f"\nEvolução temporal:")
mw_test(change_FN, change_TN, 'Taxa mudança temporal')
mw_test(trend_FN,  trend_TN,  'Tendência agrícola (slope)')

# Perfil temporal médio de classes agrícolas por ano
ag_profile_FN = []
ag_profile_TN = []
for t in range(5):
    fn_ag = np.mean([sum(1 for v in p[t].flatten().astype(int)
                         if v in AGRICULTURAL_CLASSES) / (PATCH_N**2)
                     for p in patches_FN])
    tn_ag = np.mean([sum(1 for v in p[t].flatten().astype(int)
                         if v in AGRICULTURAL_CLASSES) / (PATCH_N**2)
                     for p in patches_TN])
    ag_profile_FN.append(fn_ag)
    ag_profile_TN.append(tn_ag)

year_labels = ['T-5', 'T-4', 'T-3', 'T-2', 'T-1']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Taxa de mudança temporal
for ax, fn_v, tn_v, title, xlabel in [
    (axes[0], change_FN, change_TN,
     'Taxa de mudança temporal\n(fração pixels que mudam por ano)',
     'Fração mudança'),
    (axes[1], trend_FN, trend_TN,
     'Tendência agrícola\n(slope de pressão agrícola)',
     'Slope por ano'),
]:
    ax.hist(fn_v, bins=20, alpha=0.6, color='#c0392b',
            label=f'FN', density=True)
    ax.hist(tn_v, bins=20, alpha=0.6, color='#2980b9',
            label=f'TN', density=True)
    ax.axvline(fn_v.mean(), color='#c0392b', linestyle='--', linewidth=2)
    ax.axvline(tn_v.mean(), color='#2980b9', linestyle='--', linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Densidade')
    ax.legend()

# Perfil temporal
axes[2].plot(year_labels, ag_profile_FN, 'o-', color='#c0392b',
             linewidth=2, markersize=8, label='FN')
axes[2].plot(year_labels, ag_profile_TN, 's-', color='#2980b9',
             linewidth=2, markersize=8, label='TN')
axes[2].set_title('Pressão agrícola na vizinhança\n(T-5 a T-1)',
                  fontweight='bold')
axes[2].set_xlabel('Ano antes de T')
axes[2].set_ylabel('Fração pixels com classe agrícola')
axes[2].legend()

plt.suptitle('Evolução temporal do patch: FN vs TN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'fn_tn_temporal_evolution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: fn_tn_temporal_evolution.png")

In [ ]:
# ============================================================================
# VEREDICTO — O sinal existe nos dados?
# ============================================================================

print("=" * 65)
print("VEREDICTO DIAGNÓSTICO")
print("=" * 65)

# Coletar evidências
evidencias = []

# 1. Diferença de classes agrícolas
ag_diff = ag_FN - ag_TN
evidencias.append((
    'Classes agrícolas nos patches',
    ag_diff,
    abs(ag_diff) > 0.5,
    f'FN={ag_FN:.2f}%, TN={ag_TN:.2f}%, diff={ag_diff:+.2f}pp'
))

# 2. Entropia
ent_diff = ent_FN.mean() - ent_TN.mean()
evidencias.append((
    'Heterogeneidade (entropia)',
    ent_diff,
    p_ent < 0.05,
    f'FN={ent_FN.mean():.4f}, TN={ent_TN.mean():.4f}, p={p_ent:.4f}'
))

# 3. Vizinhos diferentes do centro
cd_diff = cdiff_FN.mean() - cdiff_TN.mean()
evidencias.append((
    'Contraste centro vs vizinhança',
    cd_diff,
    p_cd < 0.05,
    f'FN={cdiff_FN.mean():.4f}, TN={cdiff_TN.mean():.4f}, p={p_cd:.4f}'
))

# 4. Taxa de mudança temporal
ch_diff = change_FN.mean() - change_TN.mean()
_, p_ch = stats.mannwhitneyu(change_FN, change_TN, alternative='two-sided')
evidencias.append((
    'Mudança temporal no patch',
    ch_diff,
    p_ch < 0.05,
    f'FN={change_FN.mean():.4f}, TN={change_TN.mean():.4f}, p={p_ch:.4f}'
))

# 5. Tendência agrícola
_, p_tr = stats.mannwhitneyu(trend_FN, trend_TN, alternative='two-sided')
tr_diff = trend_FN.mean() - trend_TN.mean()
evidencias.append((
    'Tendência agrícola crescente',
    tr_diff,
    p_tr < 0.05,
    f'FN={trend_FN.mean():.6f}, TN={trend_TN.mean():.6f}, p={p_tr:.4f}'
))

n_significativas = sum(1 for _, _, sig, _ in evidencias if sig)

print()
for nome, diff, sig, detalhe in evidencias:
    marcador = '✅' if sig else '⚠️ '
    print(f"  {marcador} {nome}")
    print(f"     {detalhe}")
    print()

print("=" * 65)
print(f"Evidências significativas: {n_significativas}/{len(evidencias)}")
print("=" * 65)

if n_significativas >= 3:
    print("""
DIAGNÓSTICO: ERRO DE REPRESENTAÇÃO

FN e TN são distinguíveis nos patches antes de T.
O sinal existe nos dados — o modelo não está lendo corretamente.

→ Re-treino com curriculum de erros é justificado.
→ Features de delta temporal e contraste espacial devem ajudar.
→ Próximo passo: implementar representação de mudança no patch.
""")
elif n_significativas >= 1:
    print("""
DIAGNÓSTICO: SINAL FRACO MAS PRESENTE

Algumas diferenças existem mas são pequenas.
O sinal está parcialmente nos dados — mas é difícil de capturar.

→ Vale tentar features de delta temporal.
→ Expectativa de ganho moderado.
""")
else:
    print("""
DIAGNÓSTICO: ERRO EPISTÊMICO

FN e TN são indistinguíveis nos patches antes de T.
O sinal não está disponível na janela temporal observada.

→ Re-treino com curriculum de erros não vai ajudar.
→ Precisamos de features de contexto externo:
   - Taxa histórica de conversão regional
   - Distância à fronteira agrícola
   - Infraestrutura (estradas, silos) nas proximidades
""")

print(f"Gráficos salvos em: {FREEZE_DIR}")